# 008 Non-Self Retrieval BGE Dev Evaluation

This notebook reviews dev-only BGE retrieval scoring against the non-self fixture. It does not score holdout, train, or generate answers.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'data' / 'eval' / 'nonself_retrieval_bge_dev'
DEV_SUMMARY = OUT / 'retrieval_eval_summary_dev_bge_small_en_v15.json'
SAMPLE_SUMMARY = OUT / 'retrieval_eval_summary_sample_bge_small_en_v15.json'

SUMMARY = DEV_SUMMARY if DEV_SUMMARY.exists() else SAMPLE_SUMMARY
SUMMARY_MD = SUMMARY.with_suffix('.md')
prefix = 'dev' if 'summary_dev' in SUMMARY.name else 'sample'

METRICS = OUT / f'retrieval_metrics_{prefix}_bge_small_en_v15.csv'
DOMAIN_METRICS = OUT / f'retrieval_domain_metrics_{prefix}_bge_small_en_v15.csv'
QUERY_DETAILS = OUT / f'retrieval_query_details_{prefix}_bge_small_en_v15.csv'
POOL_METRICS = OUT / f'retrieval_candidate_pool_metrics_{prefix}_bge_small_en_v15.csv'

assert SUMMARY.exists(), 'Run the scoring script or set RUN_FULL_EVAL/RUN_SAMPLE_EVAL true.'

Use the sample run for quick checks. Use the full run when you are ready to refresh all dev-query metrics.

In [ ]:
RUN_SAMPLE_EVAL = False
RUN_FULL_EVAL = False

if RUN_SAMPLE_EVAL:
    subprocess.run(
        [
            sys.executable,
            str(ROOT / 'scripts' / 'evaluate_nonself_retrieval_bge.py'),
            '--max-queries',
            '48',
            '--summary',
        ],
        cwd=ROOT,
        check=True,
    )

if RUN_FULL_EVAL:
    subprocess.run(
        [
            sys.executable,
            str(ROOT / 'scripts' / 'evaluate_nonself_retrieval_bge.py'),
            '--summary',
        ],
        cwd=ROOT,
        check=True,
    )

## Summary

In [ ]:
display(Markdown(SUMMARY_MD.read_text(encoding='utf-8')))
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
summary['input_counts'], summary['runtime_seconds']

## Profile Metrics

In [ ]:
metrics = pd.read_csv(METRICS)
display(metrics)

plot_metrics = metrics[metrics['relevance_scope'] == 'any_grade']
ax = plot_metrics.set_index('profile')[['hit_at_1', 'hit_at_5', 'mrr', 'ndcg']].plot.bar(
    figsize=(9, 4),
    color=['#3f6f8f', '#6c8c3f', '#9a5a44', '#705d9b'],
)
ax.set_title(f'{prefix} any-grade retrieval metrics')
ax.set_xlabel('')
ax.set_ylim(0, 1)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## Grade 2 Metrics

In [ ]:
grade_2 = metrics[metrics['relevance_scope'] == 'grade_2']
display(grade_2)

ax = grade_2.set_index('profile')[['hit_at_1', 'hit_at_5', 'mrr', 'ndcg']].plot.bar(
    figsize=(9, 4),
    color=['#3f6f8f', '#6c8c3f', '#9a5a44', '#705d9b'],
)
ax.set_title(f'{prefix} grade-2 retrieval metrics')
ax.set_xlabel('')
ax.set_ylim(0, 1)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## Domain Metrics

In [ ]:
domain_metrics = pd.read_csv(DOMAIN_METRICS)
display(domain_metrics.head(18))

domain_any = domain_metrics[domain_metrics['relevance_scope'] == 'any_grade']
pivot = domain_any.pivot_table(
    index='primary_domain',
    columns='profile',
    values='hit_at_5',
)
display(pivot)

ax = pivot.plot.bar(figsize=(10, 4), color=['#3f6f8f', '#3d7b67', '#9a5a44'])
ax.set_title(f'{prefix} hit@5 by domain, any-grade')
ax.set_xlabel('')
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Candidate Pools

In [ ]:
pool_metrics = pd.read_csv(POOL_METRICS)
display(pool_metrics)

ax = pool_metrics.set_index('profile')['median_candidate_pool_size'].plot.bar(
    figsize=(8, 3.5),
    color='#3d7b67',
)
ax.set_title(f'{prefix} median candidate pool size')
ax.set_xlabel('')
ax.set_ylabel('docs')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## Query Details

In [ ]:
details = pd.read_csv(QUERY_DETAILS)
cols = [
    'profile',
    'query_id',
    'expected_primary_domain',
    'triage_predicted_domains',
    'candidate_pool_size',
    'first_any_grade_rank',
    'first_grade_2_rank',
    'top_relevance_grade',
]
display(details[cols].head(20))
misses = details[(details['profile'] == 'flat_bge_small_en_v15') & details['first_any_grade_rank'].isna()]
display(misses[cols].head(20))

## Exit Criteria

- Use sample metrics only for smoke checking.
- Use full dev metrics before comparing against another embedding model.
- Do not score holdout until the dev chart story is stable.
- Next step after full dev review: run the same scorer with the Qwen3 embedding challenger.